In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con).to_string(index=False))

In [ ]:
q1 = "SELECT COUNT(*) AS rows_total, SUM(analysed) AS analysed FROM matches"
print(pd.read_sql(q1, con).to_string(index=False))

q2 = """
SELECT t.name, COUNT(*) AS matches
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1 AND m.file_year >= 2015
GROUP BY t.name
ORDER BY matches DESC
"""
print(pd.read_sql(q2, con).to_string(index=False))

In [ ]:
#matches per year and per surface
q3 = """
SELECT file_year, surface, COUNT(*) AS matches
FROM matches
WHERE file_year >= 2015
GROUP BY file_year, surface
ORDER BY file_year, surface
"""
print(pd.read_sql(q3, con).to_string(index=False))

In [ ]:
# same as above but in a table 
df3 = pd.read_sql(q3, con)
print(df3.pivot(index="file_year", columns="surface", values="matches").to_string())

In [ ]:
#percentage of service points that ended in ace
q4 = """
SELECT t.name,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name
ORDER BY aces_pct DESC
"""
print(pd.read_sql(q4, con).to_string(index=False))

In [ ]:
#same above but evolution by years
q5 = """
SELECT t.name,
       m.file_year,
       COUNT(*) AS matches_used,
       ROUND(100.0 * SUM(m.w_ace + m.l_ace) / SUM(m.w_svpt + m.l_svpt), 2) AS aces_pct
FROM matches m
JOIN tournaments t ON m.event_no = t.event_no
WHERE m.analysed = 1
  AND m.file_year >= 2015
  AND m.w_ace IS NOT NULL AND m.l_ace IS NOT NULL
  AND m.w_svpt IS NOT NULL AND m.l_svpt IS NOT NULL
GROUP BY t.name, m.file_year
HAVING COUNT(*) >= 20
ORDER BY t.name, m.file_year
"""
df5 = pd.read_sql(q5, con)

print(df5.pivot(index="name", columns="file_year", values="aces_pct").to_string())
print()
print(df5.pivot(index="name", columns="file_year", values="matches_used").to_string())